# Agent란?
- LLM이 스스로 판단해서 어떤 행동(룰 사용 포함)을 할 지 결정하는 실행주체를 의미합니다.

In [43]:
from dotenv import load_dotenv
from langchain_openai.chat_models.base import ChatOpenAI
import os

load_dotenv()
print(os.environ.get('OPENAI_API_KEY')[:20])

sk-proj-GGHq2YjvxskA


Agent를 만들어야하는 이유

In [56]:
prompt = "cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $522.00 + $76.16 + $29.12"

In [57]:
chat = ChatOpenAI(temperature=0.1)

In [58]:
result = chat.invoke(prompt)

In [59]:
result.content

'The total cost is $4,363.38.'

In [60]:
chat = ChatOpenAI(model="gpt-4o", temperature=0.1)

In [61]:
result = chat.invoke(prompt)

In [62]:
result.content

'To find the total cost, you need to add all the amounts together:\n\n\\[ \n355.39 + 924.87 + 721.2 + 1940.29 + 573.63 + 65.72 + 35.00 + 522.00 + 76.16 + 29.12 = 5243.38 \n\\]\n\nSo, the total cost is $5243.38.'

## Agent 생성

In [44]:
from langchain.agents import create_agent
from langchain.tools import tool

In [23]:
@tool
def plus(num1: float, num2: float) -> float:
    """
        Adds two numbers and return the result.
    """
    return num1 + num2


"""
    json {
        "name": plus,
        "description": "Adds two numbers and return the result.",
        "parameters": {
            "num1": "float",
            "num2": "float",
        }
    
    }

"""

None

In [24]:
agent = create_agent(
    model="gpt-3.5-turbo",
    tools=[plus],
    system_prompt="You are a helpful assistant"
)

In [25]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": prompt
        }
    ]
})

In [26]:
result

{'messages': [HumanMessage(content='cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $522.00 + $76.16 + $29.12', additional_kwargs={}, response_metadata={}, id='39190db5-57a9-4bf6-ad85-7114d977c5bc'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 127, 'prompt_tokens': 109, 'total_tokens': 236, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-Dh72tGVZphi7yXTYxadDTsafDfden', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e3ea6-7e44-7cf2-b33c-f8772dc90881-0', tool_calls=[{'name': 'plus', 'args': {'num1': 355.39, 'num2': 924.87}, 'id': 'call_TZApC3tX6hB3IA62UyNaEBLg', 'type': 'tool_call'},

In [27]:
for message in result["messages"]:
    if message.__class__.__name__ == "AIMessage" and message.tool_calls:
        for i in message.tool_calls:
            print(i)

{'name': 'plus', 'args': {'num1': 355.39, 'num2': 924.87}, 'id': 'call_TZApC3tX6hB3IA62UyNaEBLg', 'type': 'tool_call'}
{'name': 'plus', 'args': {'num1': 721.2, 'num2': 1940.29}, 'id': 'call_BvsKChzXURunaYx5J7fkgduO', 'type': 'tool_call'}
{'name': 'plus', 'args': {'num1': 573.63, 'num2': 65.72}, 'id': 'call_2m1EbqmYjwJ6PWrE8CC4ONnt', 'type': 'tool_call'}
{'name': 'plus', 'args': {'num1': 35, 'num2': 522}, 'id': 'call_1sX8ZDAvqqGExD9d1pvOUJxT', 'type': 'tool_call'}
{'name': 'plus', 'args': {'num1': 76.16, 'num2': 29.12}, 'id': 'call_UfRgNggUdx8qwNEN5K2Nmlcj', 'type': 'tool_call'}


In [28]:
@tool
def total_sum(numbers: list[float]) -> float:
    """    
        Adds a list of numbers and returns the total sum.
        Use this tool when you need to calculate the total of multiple numbers.
        Input should be a string representation of a list.
        Example: "[1, 2, 3]"
    """

    return sum(numbers)

In [29]:
agent = create_agent(
    model="gpt-3.5-turbo",
    tools=[plus],
    system_prompt="You are a helpful assistant"
)

In [30]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": prompt
        }
    ]
})

In [ ]:
result["messages"][-1].content

In [ ]:
for message in result["messages"]:
    if message.__class__.__name__ == "AIMessage" and message.tool_calls:
        for i in message.tool_calls:
            print(i)

## 랭스미스

# LangSmith(랭스미스)
- LLM 기반 애플리케이션의 디버깅, 성능 평가, 모니터링 등을 제공하는 랭체인의 통합 플랫폼입니다.

## 랭스미스 Open API Key 발급
- https://smith.langchain.com/ 접속
- 로그인 후 좌측 하단 [Setting] 메뉴 클릭
- [API Keys] 클릭 후 생성
- Description은 lang_ksh(이니셜)
- [default workspace]는 기존에 있는 workspace1로 만들고 생성
- 발급받은 KEY를 .env에 추가하기

### .env에 추가하기
- LANGCHAIN_TRACING_V2=true
- LANGCHAIN_ENDPOINT="https://api.smith.langchain.com"
- LANGCHAIN_PROJECT=lang_1900
- LANGSMITH_API_KEY=발급받은 랭스미스 key

### 설정 후 Jupyter Notebook 재실행

## Agent가 동작하는 과정
1. 끝날때까지 반복이 되는 loop입니다.
2. llm으로 부터 어떤 것을 할지(get action)을 받아온다. (lang smith의 output에서 확인가능)
3. 실행한 결과를 observation이라고 부른다. 다시 다음 next action을 실행시킨다.
4. Agent Finish를 응답받으면 마지막 action 값을 리턴한다.

### 1. ReAct Agent

In [45]:
from dotenv import load_dotenv
import os

from langchain_core.prompts import ChatPromptTemplate
from langchain.tools import tool, BaseTool
from langchain_classic.agents import AgentExecutor, create_react_agent, create_openai_functions_agent
from langchain_classic import hub
from langchain.agents import create_agent
from langchain_openai.chat_models.base import ChatOpenAI

from pydantic import BaseModel, Field
from typing import Any, Type, List #Python의 내장 모듈 typing

In [46]:
load_dotenv()

True

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
@tool
def plus(expression: str) -> float:
    """
        Adds multiple numbers and returns their total sum.

        The input must be a comma-spreated string of numbers.
        Example: "10,20,30"

        Use this tool when you need to calulate the sum of multiple values.
    """
    try:
        numbers = [float(num) for i in experssion.split(",")]
        return sum(numbers)
    except Exception as e:
        return -1

In [ ]:
tools = [plus]
react_agent_prompt = hub.pull("hwchase17/react")
# 판단
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=react_agent_prompt
)

# 실행기
react_agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True # 내부 동작 확인
)

### 2. OpenAI Function Calling Agent

In [ ]:
class CalculatorToolArgsSchema(BaseModel):
    numbers: List[float] = Field(description="Numbers to sum")

class CalculatorTool(BaseTool):
    # 약속된 필드 이름
    name: Type[str] = "calculator_tool"
    description: Type[str] = """
        Adds multiple numbers and returns their total sum.
        Use this tool when you need to calulate the sum of multiple values.
    """
    args_schema: Type[BaseModel] = CalculatorToolArgsSchema
    
    # BaseTool은 반드시 _run 함수를 재정의
    # tool을 호출했을 때 실행되는 메인로직
    def _run(self, numbers):
        return sum(numbers)

In [ ]:
tools = [CalculatorTool()]

# placeholder(agent_scratchpad): 내부 tool, reasoning 호출 기록을 임시 저장
function_agent_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("placeholder", """{agent_scratchpad}"""),
])

agent = create_openai_functions_agent(
    llm=llm,
    tools=tools,
    prompt=function_agent_prompt
)

# 실행기
calling_agent_excutor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

# v1.0 create_agent(커스텀 툴)

In [75]:
from dotenv import load_dotenv
import os
import requests

from langchain_core.prompts import ChatPromptTemplate
from langchain.tools import tool, BaseTool
from langchain.agents import create_agent
from langchain_openai.chat_models.base import ChatOpenAI

from pydantic import BaseModel, Field
from typing import Any, Type, List, Tuple, Dict #Python의 내장 모듈 typing

# 추가된 import
from langchain_community.utilities.duckduckgo_search import DuckDuckGoSearchAPIWrapper
from geopy.geocoders import Nominatim

load_dotenv()
print(os.environ.get('OPENAI_API_KEY')[:20])

sk-proj-GGHq2YjvxskA


In [48]:
tools = []

agent = create_agent(
    model="gpt-4o-mini",
    tools=tools,
)

In [63]:
prompt = ChatPromptTemplate.from_messages([
    ("human", "강남의 현재 실시간 날씨 알려줘!")
])

chain = prompt | agent
result = chain.invoke({})

result

{'messages': [HumanMessage(content='강남의 현재 실시간 날씨 알려줘!', additional_kwargs={}, response_metadata={}, id='c694a101-a232-4c2e-bb93-661d9d6b543d'),
  AIMessage(content='죄송하지만, 현재 실시간 날씨 정보를 제공할 수는 없습니다. 강남의 현재 날씨를 알고 싶으시다면, 기상 웹사이트나 날씨 앱을 확인하시는 것이 좋습니다. 도움이 필요하시면 다른 질문을 해 주세요!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 18, 'total_tokens': 73, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_da1f8e43b9', 'id': 'chatcmpl-Dh7DckG4IYU4Zd4I0wFGpmXzOJQCs', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e3eb0-a21c-79e2-9b18-e69d88945109-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 55, 'tota

In [78]:
def get_coordinates(location_name):
    locator = Nominatim(user_agent="mys")
    location = locator.geocode(location_name)

    return location.latitude, location.longitude
        

In [79]:
get_coordinates("강남")

(37.4979497, 127.0275574)

In [81]:
# 위도, 경도 -> 날씨
def get_weather(lat, lon):
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
    
    # 기상 코드(WMO Code)를 한글로 변환하는 딕셔너리
    weather_codes = {
        0: "맑음 ☀️",
        1: "대체로 맑음 🌤️", 2: "구름 조금 ⛅", 3: "흐림 ☁️",
        45: "안개 🌫️", 48: "침강 안개 🌫️",
        51: "가벼운 이슬비 🌦️", 53: "이슬비 🌧️", 55: "강한 이슬비 ⛈️",
        61: "약한 비 💧", 63: "보통 비 ☔", 65: "강한 비 🌊",
        71: "약한 눈 ❄️", 73: "보통 눈 ☃️", 75: "강한 눈 🏔️",
        80: "약한 소나기 🌦️", 81: "보통 소나기 🌧️", 82: "강한 소나기 ⛈️",
        95: "뇌우 ⚡", 96: "뇌우 및 우박 ⛈️", 99: "심한 뇌우 🌪️"
    }

    try:
        response = requests.get(url)
        datas = response.json()

        if "current_weather" in datas:
            current = datas["current_weather"]
            temp = current["temperature"]
            wind = current["windspeed"]
            code = current["weathercode"]

            condition = weather_codes.get(code, "알 수 없음")

            return f"상태: {condition}\n온도: {temp}°C\n풍속: {wind}km/h"
        
    except Exception as e:
        return "요청 실패"

In [82]:
print(get_weather(37.500078, 127.035548))

상태: 흐림 ☁️
온도: 23.0°C
풍속: 5.6km/h


## 함수 -> 툴로 변경 후 제공

In [83]:
class CoordinatesToolArgSchema(BaseModel):
    location_name: str = Field("위도와 경도로 바꾸고 싶은 장소명입니다.")

class CoordinatesTool(BaseTool):
    name: Type[str] = "coordinates_tool"
    description: Type[str] = """
        장소명을 위도(latitude)와 경도(longitude) 좌표로 변환합니다.
        장소명을 위도와 경도로 변환하고 싶을 대 사용하는 도구입니다.
    """
    args_schema: Type[BaseModel] = CoordinatesToolArgSchema

    def _run(self, location_name: str) -> Tuple[float, float]:
        locator = Nominatim(user_agent="ksh")
        location = locator.geocode(location_name)
    
        return location.latitude, location.longitude, 

In [84]:
class WeatherSearchToolArgSchema(BaseModel):
    lat: float = Field(description="위도, Example Value: 37.500078")
    lon: float = Field(description="경도, Example Value: 127.035548")
    
class WeatherSearchTool(BaseTool):
    name: Type[str] = "weather_search_tool"
    description: Type[str] = """
        지역의 날씨를 가져오고 싶을 때 사용하는 툴입니다.
        위도와 경도를 입력하면, 해당 지역의 날씨의 정보를 문자열로 반환합니다.
    """

    args_schema: Type[BaseModel] = WeatherSearchToolArgSchema
    
    # 위도, 경도 -> 날씨
    def _run(self, lat: float, lon: float) -> str:
        url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
        
        # 기상 코드(WMO Code)를 한글로 변환하는 딕셔너리
        weather_codes = {
            0: "맑음 ☀️",
            1: "대체로 맑음 🌤️", 2: "구름 조금 ⛅", 3: "흐림 ☁️",
            45: "안개 🌫️", 48: "침강 안개 🌫️",
            51: "가벼운 이슬비 🌦️", 53: "이슬비 🌧️", 55: "강한 이슬비 ⛈️",
            61: "약한 비 💧", 63: "보통 비 ☔", 65: "강한 비 🌊",
            71: "약한 눈 ❄️", 73: "보통 눈 ☃️", 75: "강한 눈 🏔️",
            80: "약한 소나기 🌦️", 81: "보통 소나기 🌧️", 82: "강한 소나기 ⛈️",
            95: "뇌우 ⚡", 96: "뇌우 및 우박 ⛈️", 99: "심한 뇌우 🌪️"
        }
    
        try:
            response = requests.get(url)
            datas = response.json()
    
            if "current_weather" in datas:
                current = datas["current_weather"]
                temp = current["temperature"]
                wind = current["windspeed"]
                code = current["weathercode"]
    
                condition = weather_codes.get(code, "알 수 없음")
    
                return f"상태: {condition}\n온도: {temp}°C\n풍속: {wind}km/h"
            
        except Exception as e:
            return "요청 실패"

In [85]:
tools= [CoordinatesTool(), WeatherSearchTool()]

agent = create_agent(
    model="gpt-4o-mini",
    tools=tools,
)

prompt = ChatPromptTemplate.from_messages([
    ("human", "강남의 현재 실시간 날씨 알려줘!")
])

chain = prompt | agent
result = chain.invoke({})

result

{'messages': [HumanMessage(content='강남의 현재 실시간 날씨 알려줘!', additional_kwargs={}, response_metadata={}, id='4774ac4e-7720-41cc-8c28-bf9c346e7500'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 185, 'total_tokens': 201, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_9ed8b4905a', 'id': 'chatcmpl-Dh7qKAzaMbmxxl0m5Y2FtXG1LqcqT', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e3ed5-3e85-7063-9bdc-24f38a1f2eae-0', tool_calls=[{'name': 'coordinates_tool', 'args': {'location_name': '강남'}, 'id': 'call_baPGyowsTj4GnK3TyJX3ruAo', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 185, 'out

## Stock Agent

In [1]:
from dotenv import load_dotenv
import os
import requests

from langchain_core.prompts import ChatPromptTemplate
from langchain.tools import tool, BaseTool
from langchain.agents import create_agent
from langchain_openai.chat_models.base import ChatOpenAI

from pydantic import BaseModel, Field
from typing import Any, Type, List, Tuple, Dict #Python의 내장 모듈 typing

# 추가된 import
from langchain_community.utilities.duckduckgo_search import DuckDuckGoSearchAPIWrapper
from geopy.geocoders import Nominatim

load_dotenv()
print(os.environ.get('OPENAI_API_KEY')[:20])

sk-proj-GGHq2YjvxskA


-  https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol=IBM&apikey=demo